In [67]:
import geopandas as gpd, folium
from IPython.display import display, HTML

In [49]:
ABT = gpd.read_file("../../../../Data/Final_dataset/ABT/ABT.gpkg", layer="subdivisions")
# tax_parcel_cama_dataset = gpd.read_file("../../../../Data/Original_dataset/original.gdb", layer = "Tax_CAMA_1_25", columns=["heatedarea", "totalarea", "geometry"]).to_crs(ABT.crs)

In [50]:
ABT = ABT.loc[(ABT["FAR_total"] == 0)]

In [51]:
tax_parcel_cama_dataset["geometry"] = tax_parcel_cama_dataset.geometry.buffer(0)
ABT["geometry"] = ABT.geometry.buffer(0)

In [52]:
# Ensure projected CRS (area-safe)
assert ABT.crs.is_projected, "CRS must be projected for area calculations"

# Parcel area
parcels = tax_parcel_cama_dataset.copy()
parcels["parcel_area"] = parcels.geometry.area

# TRUE geometric intersection (clips geometries)
parcel_abt_intersection = gpd.overlay(
    parcels,
    ABT[["subd_id", "year", "geometry"]],
    how="intersection"
)

# Overlap area and ratio
parcel_abt_intersection["overlap_area"] = parcel_abt_intersection.geometry.area
parcel_abt_intersection["overlap_ratio"] = (
    parcel_abt_intersection["overlap_area"]
    / parcel_abt_intersection["parcel_area"]
)

# Majority rule: retain parcels with >50% overlap
parcels_in_abt_majority = parcel_abt_intersection.loc[
    parcel_abt_intersection["overlap_ratio"] > 0.5
].copy()

C:\Users\erfan\AppData\Local\Programs\Python\Python311\Lib\site-packages\geopandas\tools\overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 198 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


In [53]:
ABT_wgs = ABT.to_crs(epsg=4326)
parcels_wgs = parcels_in_abt_majority.to_crs(epsg=4326)
center = [ABT_wgs.geometry.centroid.y.mean(), ABT_wgs.geometry.centroid.x.mean()]

C:\Users\erfan\AppData\Local\Temp\ipykernel_20920\3940373370.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = [ABT_wgs.geometry.centroid.y.mean(), ABT_wgs.geometry.centroid.x.mean()]


In [85]:
m = folium.Map(
    location=center,
    zoom_start=12,          # start closer
    min_zoom=6,
    max_zoom=22,            # allow deep zoom
    tiles=None,
    control_scale=True,
    prefer_canvas=True      # better performance at high zoom
)

In [86]:
# Satellite imagery (Esri World Imagery)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery",
    name="Satellite Imagery",
    overlay=False,
    control=True,
    max_zoom=22,
    max_native_zoom=19,     # Esri native resolution
    zoom_offset=0,
    detect_retina=True
).add_to(m)


# Imagery metadata (Citation) layer — this contains date & source info
folium.TileLayer(
    tiles="https://services.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/4/tile/{z}/{y}/{x}",
    attr="Esri Imagery Metadata",
    name="Imagery Metadata",
    overlay=True,
    control=True
).add_to(m)

# Labels overlay
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Reference/World_Boundaries_and_Places/MapServer/tile/{z}/{y}/{x}",
    attr="Esri Reference",
    name="Labels",
    overlay=True,
    control=True
).add_to(m)

In [87]:
folium.GeoJson(
    ABT_wgs,
    name="ABT Subdivisions (FAR = 0)",
    style_function=lambda x: {
        "fillOpacity": 0.0,
        "color": "#000000",
        "weight": 2,
        "dashArray": "5,2",
        "opacity": 1.0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["subd_id", "year"],
        aliases=["Subdivision ID:", "Year:"],
        sticky=True,
        labels=True,
        style=(
            "background-color: white; "
            "border: 1px solid black; "
            "border-radius: 3px; "
            "padding: 4px; "
            "font-size: 11px;"
        )
    )
).add_to(m)


for _, row in ABT_wgs.iterrows():
    c = row.geometry.centroid
    folium.Marker(
        location=[c.y, c.x],
        icon=folium.DivIcon(
            html=f"""
            <div style="
                font-size: 10px;
                color: black;
                background-color: rgba(255,255,255,0.85);
                border: 1px solid black;
                border-radius: 3px;
                padding: 2px 4px;
                text-align: center;
                white-space: nowrap;
            ">
                {int(row.year)}
            </div>
            """
        )
    ).add_to(m)


In [88]:
folium.GeoJson(
    parcels_wgs,
    name="Parcels in ABT",
    style_function= lambda x: {
        "fillOpacity": 0.0,        # 🔴 NO fill
        "color": "#ffff00",        # black edge
        "weight": 2,              # VERY bold
        "dashArray": "5,2",       # strong dash
        "opacity": 1.0             # force visibility
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["heatedarea", "totalarea"],
        aliases=["Heated Area:", "Total Area:"],
        sticky=True
    )
).add_to(m)

In [89]:
folium.LayerControl(collapsed=False).add_to(m)
m

In [90]:
def show_html_table(df, max_height=650, max_width="100%", decimals=3):
    df2 = df.copy()

    # If it's a GeoDataFrame, drop geometry for display
    if "geometry" in df2.columns:
        df2 = df2.drop(columns="geometry")

    # Light rounding for readability
    num_cols = df2.select_dtypes(include="number").columns
    df2[num_cols] = df2[num_cols].round(decimals)

    html = df2.to_html(index=False)

    # Wrap table in a scrollable div
    html = f"""
    <div style="max-height:{max_height}px; overflow:auto; border:1px solid #ddd; width:{max_width};">
        {html}
    </div>
    """

    display(HTML(html))

# Example: show full ABT (careful if huge)
show_html_table(ABT)


subd_id,issue_date,year,HAC_dist,BAD,SHD,int_den025,nd_deg025,int_den05,nd_deg05,int_den075,nd_deg075,int_den1,nd_deg1,AI,PROX,ENN_MN,ED,SHAPE_MN,FRAC_MN,ENN_inv,ED_inv,SHAPE_inv,FRAC_inv,AI_norm,PROX_norm,ENN_inv_norm,ED_inv_norm,SHAPE_inv_norm,FRAC_inv_norm,COMPACTNESS_SUM,BAD_ctx_025,BAD_ctx_050,FAR,groceries_ws,transit_ws,FAR_total
212,2018,2018.0,1.45,0.591,0.50,0.169,2.237,0.100,2.171,0.131,2.240,0.107,2.301,0.941,43215.000,7.571,704.569,2.237,1.179,0.132,0.001,0.447,0.848,0.892,0.089,0.130,0.003,0.214,0.492,0.091,0.139,0.146,0.0,25.91,28.0,0.0
545,1990,1990.0,3.56,0.194,0.00,0.124,2.176,0.104,2.240,0.093,2.344,0.088,2.463,0.512,589.725,12.828,1448.117,1.170,1.071,0.078,0.001,0.855,0.934,0.110,0.001,0.076,0.001,0.775,0.747,0.033,0.106,0.109,0.0,4.02,37.0,0.0
1421,1990,1990.0,4.88,0.151,0.20,0.046,2.095,0.043,2.360,0.064,2.417,0.052,2.378,0.820,4838.980,35.492,428.996,1.379,1.103,0.028,0.002,0.725,0.907,0.671,0.009,0.026,0.005,0.597,0.667,0.024,0.079,0.127,0.0,77.28,36.0,0.0
1805,2004,2004.0,3.71,0.004,0.00,0.124,2.280,0.106,2.460,0.119,2.433,0.118,2.508,0.824,600.073,45.777,18.662,1.025,1.011,0.022,0.054,0.975,0.989,0.678,0.001,0.020,0.141,0.942,0.913,0.053,0.085,0.093,0.0,4.89,31.0,0.0
2578,1990,1990.0,7.44,0.081,0.00,0.067,2.000,0.036,2.044,0.022,2.035,0.014,2.032,0.483,722.839,14.460,578.137,1.341,1.130,0.069,0.002,0.746,0.885,0.055,0.001,0.067,0.004,0.626,0.601,0.029,0.046,0.024,0.0,0.00,0.0,0.0
2657,2002,2002.0,1.98,0.061,0.00,0.116,2.095,0.118,2.183,0.112,2.205,0.100,2.242,0.514,684.845,12.171,451.258,1.266,1.108,0.082,0.002,0.790,0.902,0.113,0.001,0.080,0.005,0.686,0.653,0.035,0.084,0.099,0.0,36.08,30.0,0.0
3456,1986,1986.0,7.23,0.082,0.01,0.044,2.054,0.027,2.070,0.022,2.187,0.022,2.252,0.549,1044.320,15.938,494.043,1.297,1.112,0.063,0.002,0.771,0.899,0.176,0.002,0.061,0.004,0.660,0.645,0.029,0.104,0.081,0.0,0.00,0.0,0.0
3583,2003,2003.0,8.02,0.050,0.00,0.033,1.889,0.038,2.125,0.061,2.262,0.061,2.244,0.470,590.730,13.667,365.098,1.326,1.124,0.073,0.003,0.754,0.889,0.032,0.001,0.071,0.006,0.637,0.615,0.031,0.026,0.046,0.0,0.00,16.0,0.0
4108,2015,2015.0,3.04,0.248,0.00,0.051,2.211,0.056,2.373,0.061,2.463,0.060,2.526,0.920,11879.138,21.041,418.245,1.248,1.057,0.048,0.002,0.801,0.946,0.854,0.024,0.046,0.005,0.702,0.784,0.039,0.111,0.091,0.0,20.22,32.0,0.0
4532,2005,2005.0,3.01,0.248,0.63,0.126,2.235,0.146,2.266,0.130,2.344,0.127,2.346,0.830,2900.000,1.000,1061.494,1.505,1.146,1.000,0.001,0.664,0.873,0.689,0.005,1.000,0.001,0.514,0.565,0.358,0.116,0.120,0.0,4.44,22.0,0.0


In [91]:
!jupyter nbconvert --to html --no-input temp_FAR.ipynb --output ../../../../output/Notebook_Outputs/FAR0.html

[NbConvertApp] Converting notebook temp4.ipynb to html
[NbConvertApp] Writing 982918 bytes to ..\..\..\..\output\Notebook_Outputs\FAR0.html
